# Knowledge-Guided NILM — Phase 2: CNN + BiLSTM + Attention-Gated Knowledge Fusion
**Dataset:** UK-DALE 2017, House 1
**Appliances:** Dishwasher, Washing Machine, Laptop, Fridge, Office Lamp 1
(kettle was swapped out in Phase 1 — see Phase 1 notebook — due to near-zero real usage
events at this house making it unlearnable; this notebook keeps the Phase 1 appliance set)

**Architecture (per the CNN + Knowledge Vector design report):**
1. 1D-CNN branch — learns local waveform shape from the raw aggregate window
2. Knowledge extractor branch — 12 hand-engineered electrical features per window
3. BiLSTM + temporal attention — models how CNN features evolve across the window
4. Attention-gated knowledge fusion — knowledge embedding gates the CNN context, then
   both are concatenated
5. Multi-task heads — (a) appliance state (multi-label ON/OFF), (b) duty-cycle
   regression (fraction of window each appliance was ON), (c) knowledge-vector
   reconstruction (auxiliary loss keeping the fused representation physically meaningful)
6. Post-hoc minimum-dwell-time correction — removes implausible single-window flicker
   from the predicted state sequence

This notebook reuses the Phase 1 fixes already validated: shared-origin resampling,
median filtering, block-stratified (day-cycled) train/val/test split, and persistence-
based ON/OFF ground truth.


## 1. Install Dependencies

In [ ]:
!pip install -q tensorflow scikit-learn scipy pandas numpy matplotlib seaborn

## 2. Import Libraries

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import medfilt, find_peaks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix)
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## 3. Mount Google Drive & Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CONFIG = {
    "house_dir": "/content/drive/MyDrive/SEM7PROJ/house_1",  # adjust to your folder structure
    "checkpoint_dir": "/content/drive/MyDrive/SEM7PROJ/checkpoints_phase2",
    "sample_period": "6s",
    "window_size": 100,       # ~10 minutes at 6s/sample
    "window_stride": 20,
    "median_kernel": 5,
    "on_thresholds": {
        "dishwasher": 10,
        "washing machine": 20,
        "laptop": 15,
        "fridge": 20,
        "office lamp 1": 5,
    },
    # Minimum consecutive samples above threshold to count as a real ON event (denoise).
    "min_on_samples": {
        "dishwasher": 20,
        "washing machine": 10,
        "laptop": 5,
        "fridge": 5,
        "office lamp 1": 3,
    },
    # Window labeling rule per appliance: "majority" = ON if >50% of window is in the
    # persistent ON state (good for long-duration appliances); "any" = ON if the
    # appliance appears at all in the window (better for very short-burst appliances
    # whose events are shorter than a typical window — this was the fix the design
    # report used for kettle; none of the current 5 appliances need it, but it's left
    # configurable here in case a future appliance swap needs it).
    "label_rule": {
        "dishwasher": "majority",
        "washing machine": "majority",
        "laptop": "majority",
        "fridge": "majority",
        "office lamp 1": "majority",
    },
    "appliances": ["dishwasher", "washing machine", "laptop", "fridge", "office lamp 1"],
    "batch_size": 64,
    "epochs": 50,
    "min_dwell_windows": 2,   # post-hoc correction: flip isolated runs shorter than this
    "recon_loss_weight": 0.3,
    "duty_loss_weight": 1.0,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
print("Config ready. House directory:", CONFIG["house_dir"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config ready. House directory: /content/drive/MyDrive/SEM7PROJ/house_1


## 4. Dataset Loading

In [ ]:
def load_labels(house_dir):
    labels_path = os.path.join(house_dir, "labels.dat")
    labels = {}
    with open(labels_path, "r") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                ch_num, name = parts
                labels[int(ch_num)] = name.strip().lower().replace("_", " ")
    return labels

def load_channel(house_dir, channel_num):
    path_dat = os.path.join(house_dir, f"channel_{channel_num}.dat")
    path_csv = os.path.join(house_dir, f"channel_{channel_num}.csv")
    if os.path.exists(path_dat):
        df = pd.read_csv(path_dat, sep=" ", header=None, names=["timestamp", "power"])
    elif os.path.exists(path_csv):
        df = pd.read_csv(path_csv, header=None, names=["timestamp", "power"])
    else:
        raise FileNotFoundError(f"No data file found for channel {channel_num} in {house_dir}")
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
    df = df.set_index("timestamp").sort_index()
    df["power"] = pd.to_numeric(df["power"], errors="coerce")
    return df

labels_map = load_labels(CONFIG["house_dir"])
print(f"Found {len(labels_map)} channels in House 1.")

Found 53 channels in House 1.


## 5. Appliance Selection, Shared-Origin Resampling & Aggregate Signal

In [ ]:
def find_channel_for_appliance(labels_map, appliance_name):
    appliance_name = appliance_name.lower()
    candidates = [ch for ch, name in labels_map.items()
                  if appliance_name in name or name in appliance_name]
    if not candidates:
        words = appliance_name.split()
        candidates = [ch for ch, name in labels_map.items() if all(w in name for w in words)]
    if not candidates:
        raise ValueError(f"No matching channel found for appliance '{appliance_name}'.")
    return candidates[0]

selected_channels = {}
appliance_data = {}
for app in CONFIG["appliances"]:
    ch = find_channel_for_appliance(labels_map, app)
    selected_channels[app] = ch
    df = load_channel(CONFIG["house_dir"], ch)
    appliance_data[app] = df
    print(f"{app:20s} -> channel {ch} | {len(df)} samples, {df.index.min()} to {df.index.max()}")

def resample_series(df, period, origin):
    return df["power"].resample(period, origin=origin).mean()

shared_origin = min(df.index.min() for df in appliance_data.values())
print("\nShared resample origin:", shared_origin)

resampled = {app: resample_series(df, CONFIG["sample_period"], shared_origin)
             for app, df in appliance_data.items()}

combined = pd.DataFrame(resampled).dropna(how="all")
max_ffill = int(60 / 6)
combined = combined.ffill(limit=max_ffill)
combined["aggregate"] = combined[CONFIG["appliances"]].sum(axis=1, skipna=True)

print("\nSynthetic aggregate created from:", CONFIG["appliances"])
print("Total samples:", len(combined))

dishwasher           -> channel 6 | 19819392 samples, 2012-11-09 22:28:18 to 2017-04-26 17:35:30
washing machine      -> channel 5 | 19555935 samples, 2012-11-09 22:28:18 to 2017-04-26 17:35:24
laptop               -> channel 4 | 4539118 samples, 2012-11-16 11:29:01 to 2017-04-26 17:34:48
fridge               -> channel 12 | 19381298 samples, 2012-12-14 22:21:32 to 2017-04-26 17:32:51
office lamp 1        -> channel 48 | 18179281 samples, 2013-04-11 19:08:55 to 2017-04-26 17:35:12

Shared resample origin: 2012-11-09 22:28:18

Synthetic aggregate created from: ['dishwasher', 'washing machine', 'laptop', 'fridge', 'office lamp 1']
Total samples: 22736289


## 6. Preprocessing (median filtering, missing-value handling)

In [ ]:
data = combined.dropna(subset=CONFIG["appliances"] + ["aggregate"]).copy()

def median_filter_series(series, kernel_size):
    if kernel_size % 2 == 0:
        kernel_size += 1
    return pd.Series(medfilt(series.values, kernel_size=kernel_size), index=series.index)

filtered = data.copy()
for col in CONFIG["appliances"] + ["aggregate"]:
    filtered[col] = median_filter_series(data[col], CONFIG["median_kernel"])
filtered[filtered < 0] = 0

print("Preprocessing complete. Rows remaining:", len(filtered))

Preprocessing complete. Rows remaining: 3930087


## 7. Persistent ON/OFF State & Block-Stratified Split
Reusing the Phase 1 fixes: a persistence rule denoises brief power blips before
labeling, and a day-cycled block split (7 train : 1 val : 1 test days, repeating)
guarantees every appliance has representative coverage in every split.

In [ ]:
def compute_on_mask(power, threshold, min_samples):
    raw_on = power > threshold
    if min_samples <= 1:
        return raw_on
    groups = (raw_on != raw_on.shift()).cumsum()
    run_lengths = raw_on.groupby(groups).transform("size")
    return raw_on & (run_lengths >= min_samples)

for app in CONFIG["appliances"]:
    filtered[f"{app}_on"] = compute_on_mask(
        filtered[app], CONFIG["on_thresholds"][app], CONFIG["min_on_samples"][app]
    )

filtered["date"] = filtered.index.normalize()
unique_days = filtered["date"].drop_duplicates().sort_values().reset_index(drop=True)
day_split = []
for i in range(len(unique_days)):
    r = i % 10
    day_split.append("train" if r < 7 else ("val" if r == 7 else "test"))
day_assignment = pd.Series(day_split, index=unique_days)
filtered["split"] = filtered["date"].map(day_assignment)

train_df = filtered[filtered["split"] == "train"].drop(columns=["date", "split"])
val_df = filtered[filtered["split"] == "val"].drop(columns=["date", "split"])
test_df = filtered[filtered["split"] == "test"].drop(columns=["date", "split"])

print(f"Rows -> train: {len(train_df)} | val: {len(val_df)} | test: {len(test_df)}")
print("\nAppliance ON-event coverage per split:")
for app in CONFIG["appliances"]:
    counts = {s: int(d[f"{app}_on"].sum()) for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]}
    flag = " <-- ZERO EVENTS" if 0 in counts.values() else ""
    print(f"  {app:20s} {counts}{flag}")

Rows -> train: 2739242 | val: 417176 | test: 773669

Appliance ON-event coverage per split:
  dishwasher           {'train': 74214, 'val': 11831, 'test': 18415}
  washing machine      {'train': 182825, 'val': 31428, 'test': 51911}
  laptop               {'train': 1421844, 'val': 234348, 'test': 407342}
  fridge               {'train': 1228411, 'val': 187106, 'test': 343682}
  office lamp 1        {'train': 492556, 'val': 84517, 'test': 117928}


## 8. Knowledge Vector — 12 Engineered Electrical Features (per window, aggregate-only)
Per the design report: mean, max, min, std, RMS, range of power; power jump (max
single-step derivative); transition count (number of sign changes in the derivative,
a proxy for switching activity); dominant period (via autocorrelation — captures
cyclical appliances like a compressor); rise time and fall time (samples from window
start/end to peak); duty cycle (fraction of window above a low activity floor). All
computed strictly from the aggregate signal, so nothing here requires appliance
ground truth — same no-leakage principle as Phase 1's dynamic features.

In [ ]:
def compute_knowledge_vector(window):
    w = window.astype(np.float64)
    mean_p = np.mean(w)
    max_p = np.max(w)
    min_p = np.min(w)
    std_p = np.std(w)
    rms_p = np.sqrt(np.mean(w ** 2))
    range_p = max_p - min_p

    diffs = np.diff(w)
    power_jump = np.max(np.abs(diffs)) if len(diffs) > 0 else 0.0
    # transition count: number of times the derivative changes sign (switching activity)
    signs = np.sign(diffs)
    signs = signs[signs != 0]
    transition_count = int(np.sum(np.diff(signs) != 0)) if len(signs) > 1 else 0

    # dominant period via autocorrelation: lag (in samples) of the highest non-zero-lag peak
    w_centered = w - mean_p
    if np.std(w_centered) > 1e-8:
        autocorr = np.correlate(w_centered, w_centered, mode="full")
        autocorr = autocorr[len(autocorr) // 2:]
        if len(autocorr) > 2:
            search = autocorr[1:]
            dominant_period = float(np.argmax(search) + 1)
        else:
            dominant_period = 0.0
    else:
        dominant_period = 0.0

    peak_idx = np.argmax(w)
    rise_time = float(peak_idx)                    # samples from window start to peak
    fall_time = float(len(w) - 1 - peak_idx)        # samples from peak to window end

    activity_floor = mean_p * 0.1
    duty_cycle = float(np.mean(w > activity_floor)) if mean_p > 0 else 0.0

    return np.array([mean_p, max_p, min_p, std_p, rms_p, range_p, power_jump,
                      transition_count, dominant_period, rise_time, fall_time, duty_cycle])

KNOWLEDGE_DIM = 12
print("Knowledge vector function ready (12-dim, aggregate-only, no leakage).")

Knowledge vector function ready (12-dim, aggregate-only, no leakage).


## 9. Sliding Window Generation — State Labels, Duty-Cycle Targets, Knowledge Vectors
Each window gets three targets per appliance: (a) a binary state label (via the
per-appliance `label_rule` — "majority" or "any"), (b) a duty-cycle regression target
(the true fraction of the window the appliance was ON, from the persistent mask —
this is what fixes the duration-overcounting bug described in the design report), and
(c) the 12-dim knowledge vector itself, which also serves as the reconstruction
target for the auxiliary loss.

In [ ]:
def build_windows(df, window_size, stride, appliances, label_rule):
    agg = df["aggregate"].values
    n_samples = len(agg)

    X_seq, X_know, Y_state, Y_duty = [], [], [], []
    for start in range(0, n_samples - window_size + 1, stride):
        end = start + window_size
        window_agg = agg[start:end]
        if np.isnan(window_agg).any():
            continue

        know_vec = compute_knowledge_vector(window_agg)

        state_labels, duty_targets = [], []
        for app in appliances:
            on_window = df[f"{app}_on"].values[start:end]
            duty_frac = float(np.mean(on_window))
            if label_rule[app] == "any":
                is_on = int(on_window.any())
            else:
                is_on = int(duty_frac > 0.5)
            state_labels.append(is_on)
            duty_targets.append(duty_frac)

        X_seq.append(window_agg)
        X_know.append(know_vec)
        Y_state.append(state_labels)
        Y_duty.append(duty_targets)

    return (np.array(X_seq), np.array(X_know),
            np.array(Y_state), np.array(Y_duty, dtype=np.float32))

X_train_seq, X_train_know, Y_train_state, Y_train_duty = build_windows(
    train_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_val_seq, X_val_know, Y_val_state, Y_val_duty = build_windows(
    val_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])
X_test_seq, X_test_know, Y_test_state, Y_test_duty = build_windows(
    test_df, CONFIG["window_size"], CONFIG["window_stride"], CONFIG["appliances"], CONFIG["label_rule"])

print("Train:", X_train_seq.shape, X_train_know.shape, Y_train_state.shape, Y_train_duty.shape)
print("Val:  ", X_val_seq.shape, X_val_know.shape, Y_val_state.shape, Y_val_duty.shape)
print("Test: ", X_test_seq.shape, X_test_know.shape, Y_test_state.shape, Y_test_duty.shape)

Train: (136958, 100) (136958, 12) (136958, 5) (136958, 5)
Val:   (20854, 100) (20854, 12) (20854, 5) (20854, 5)
Test:  (38679, 100) (38679, 12) (38679, 5) (38679, 5)


## 10. Dataset Preparation — Scaling (fit on TRAIN only)

In [ ]:
seq_scaler = MinMaxScaler()
X_train_seq_s = seq_scaler.fit_transform(X_train_seq.reshape(-1, 1)).reshape(X_train_seq.shape)[..., np.newaxis]
X_val_seq_s = seq_scaler.transform(X_val_seq.reshape(-1, 1)).reshape(X_val_seq.shape)[..., np.newaxis]
X_test_seq_s = seq_scaler.transform(X_test_seq.reshape(-1, 1)).reshape(X_test_seq.shape)[..., np.newaxis]

know_scaler = MinMaxScaler()
X_train_know_s = know_scaler.fit_transform(X_train_know)
X_val_know_s = know_scaler.transform(X_val_know)
X_test_know_s = know_scaler.transform(X_test_know)

with open(os.path.join(CONFIG["checkpoint_dir"], "scalers.pkl"), "wb") as f:
    pickle.dump({"seq_scaler": seq_scaler, "know_scaler": know_scaler, "config": CONFIG}, f)

print("Scaled shapes -> seq:", X_train_seq_s.shape, "| know:", X_train_know_s.shape)

Scaled shapes -> seq: (136958, 100, 1) | know: (136958, 12)


## 11. Per-Appliance Class Weights (for the state-classification loss)

In [ ]:
pos_counts = Y_train_state.sum(axis=0).astype(np.float32)
neg_counts = (Y_train_state.shape[0] - pos_counts).astype(np.float32)
# Cap kept moderate (15) rather than the design report's higher caps, since none of the
# current 5 appliances are as extreme as kettle was -- see Phase 1 notes on overcorrection.
pos_weights_arr = np.clip(neg_counts / np.maximum(pos_counts, 1.0), 1.0, 15.0)

print("Per-appliance positive-class weights:")
for app, w, pos in zip(CONFIG["appliances"], pos_weights_arr, pos_counts):
    print(f"  {app:20s} weight={w:6.2f}  (positive windows in train: {int(pos)})")

Per-appliance positive-class weights:
  dishwasher           weight= 15.00  (positive windows in train: 3797)
  washing machine      weight= 12.82  (positive windows in train: 9907)
  laptop               weight=  1.00  (positive windows in train: 71538)
  fridge               weight=  1.23  (positive windows in train: 61393)
  office lamp 1        weight=  4.56  (positive windows in train: 24652)


## 12. Model Architecture
CNN branch (local waveform shape) -> BiLSTM + temporal attention (models how CNN
features evolve across the window) -> attention-gated fusion with the knowledge
embedding -> three task heads.

In [ ]:
def build_cnn_branch(window_size):
    seq_input = layers.Input(shape=(window_size, 1), name="aggregate_sequence")
    x = layers.Conv1D(32, 5, padding="same", activation="relu")(seq_input)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return seq_input, x  # (batch, window_size, 64) sequence of local CNN features

def build_knowledge_branch(know_dim):
    know_input = layers.Input(shape=(know_dim,), name="knowledge_vector")
    k = layers.Dense(32, activation="relu")(know_input)
    k = layers.Dense(16, activation="relu")(k)
    return know_input, k  # (batch, 16)

def build_model(window_size, know_dim, n_appliances, pos_weights, recon_weight, duty_weight):
    seq_input, cnn_features = build_cnn_branch(window_size)
    know_input, know_embed = build_knowledge_branch(know_dim)

    # BiLSTM over the CNN feature sequence
    bilstm_out = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(cnn_features)
    bilstm_out = layers.Dropout(0.3)(bilstm_out)

    # Temporal attention: learn a weight per timestep, pool into one context vector
    attn_scores = layers.Dense(1, activation="tanh")(bilstm_out)          # (batch, T, 1)
    attn_weights = layers.Softmax(axis=1)(attn_scores)                    # (batch, T, 1)
    context = layers.Multiply()([bilstm_out, attn_weights])
    context = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1), name="temporal_attention_pool")(context)  # (batch, 64)

    # Attention-gated knowledge fusion: knowledge embedding produces a gate over the
    # CNN/BiLSTM context, then both are concatenated
    gate = layers.Dense(context.shape[-1], activation="sigmoid", name="knowledge_gate")(know_embed)
    gated_context = layers.Multiply(name="gated_context")([context, gate])
    fused = layers.Concatenate(name="knowledge_fusion")([gated_context, know_embed])

    shared = layers.Dense(64, activation="relu")(fused)
    shared = layers.Dropout(0.3)(shared)
    shared = layers.Dense(32, activation="relu")(shared)

    # Multi-task heads
    state_out = layers.Dense(n_appliances, activation="sigmoid", name="state")(shared)
    duty_out = layers.Dense(n_appliances, activation="sigmoid", name="duty")(shared)
    recon_out = layers.Dense(know_dim, activation="linear", name="reconstruction")(shared)

    model = models.Model(inputs=[seq_input, know_input],
                          outputs=[state_out, duty_out, recon_out],
                          name="CNN_Knowledge_BiLSTM_Attention")

    def weighted_bce(pos_weights):
        pw = tf.constant(pos_weights, dtype=tf.float32)
        def loss_fn(y_true, y_pred):
            y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
            y_true = tf.cast(y_true, tf.float32)
            bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
            return tf.reduce_mean(bce)
        return loss_fn

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss={"state": weighted_bce(pos_weights), "duty": "mse", "reconstruction": "mse"},
        loss_weights={"state": 1.0, "duty": duty_weight, "reconstruction": recon_weight},
        metrics={"state": [tf.keras.metrics.BinaryAccuracy(name="acc"),
                            tf.keras.metrics.Precision(name="precision"),
                            tf.keras.metrics.Recall(name="recall")]}
    )
    return model

model = build_model(CONFIG["window_size"], KNOWLEDGE_DIM, len(CONFIG["appliances"]),
                     pos_weights_arr, CONFIG["recon_loss_weight"], CONFIG["duty_loss_weight"])
model.summary()

Model: "CNN_Knowledge_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ aggregate_sequence  │ (None, 100, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 100, 32)   │        192 │ aggregate_sequen… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 100, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 100, 64)   │     10,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 100, 64)   │     12,352 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 64)   │        256 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 64)   │     24,832 │ batch_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 100, 64)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 100, 1)    │         65 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_vector    │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ softmax (Softmax)   │ (None, 100, 1)    │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │        416 │ knowledge_vector… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 100, 64)   │          0 │ dropout[0][0],    │
│                     │                   │            │ softmax[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        528 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ temporal_attention… │ (None, 64)        │          0 │ multiply[0][0]    │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_gate      │ (None, 64)        │      1,088 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gated_context       │ (None, 64)        │          0 │ temporal_attenti… │
│ (Multiply)          │                   │            │ knowledge_gate[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ knowledge_fusion    │ (None, 80)        │          0 │ gated_context[0]… │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 58,407 (228.15 KB)

 Trainable params: 58,087 (226.90 KB)

 Non-trainable params: 320 (1.25 KB)

## 13. Training Loop (checkpointing, early stopping, LR schedule)

In [ ]:
checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], "best_model.keras")
history_path = os.path.join(CONFIG["checkpoint_dir"], "history.json")

callback_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
]

history = model.fit(
    [X_train_seq_s, X_train_know_s],
    {"state": Y_train_state, "duty": Y_train_duty, "reconstruction": X_train_know_s},
    validation_data=(
        [X_val_seq_s, X_val_know_s],
        {"state": Y_val_state, "duty": Y_val_duty, "reconstruction": X_val_know_s}
    ),
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    callbacks=callback_list,
    verbose=1
)

with open(history_path, "w") as f:
    json.dump(history.history, f)
print("Training complete.")

Epoch 1/50
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - duty_loss: 0.0720 - loss: 0.5029 - reconstruction_loss: 0.0638 - state_acc: 0.8602 - state_loss: 0.4118 - state_precision: 0.6719 - state_recall: 0.8679
Epoch 1: val_loss improved from None to 1.74267, saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2/best_model.keras
2140/2140 ━━━━━━━━━━━━━━━━━━━━ 71s 27ms/step - duty_loss: 0.0530 - loss: 0.3957 - reconstruction_loss: 0.0341 - state_acc: 0.8892 - state_loss: 0.3324 - state_precision: 0.7197 - state_recall: 0.9124 - val_duty_loss: 0.2242 - val_loss: 1.7427 - val_reconstruction_loss: 0.0373 - val_state_acc: 0.5794 - val_state_loss: 1.5074 - val_state_precision: 0.2815 - val_state_recall: 0.3776 - learning_rate: 0.0010
Epoch 2/50
2139/2140 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - duty_loss: 0.0417 - loss: 0.3116 - reconstruction_loss: 0.0132 - state_acc: 0.9142 - state

## 14. Validation

In [ ]:
val_results = model.evaluate(
    [X_val_seq_s, X_val_know_s],
    {"state": Y_val_state, "duty": Y_val_duty, "reconstruction": X_val_know_s},
    verbose=1
)
print("\nValidation results:")
for name, value in zip(model.metrics_names, val_results):
    print(f"  {name}: {value:.4f}")

652/652 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - duty_loss: 0.0588 - loss: 0.5026 - reconstruction_loss: 0.0062 - state_acc: 0.9069 - state_loss: 0.4422 - state_precision: 0.8445 - state_recall: 0.7953

Validation results:
  loss: 0.5026
  compile_metrics: 0.4422
  state_loss: 0.0588
  duty_loss: 0.0062
  reconstruction_loss: 0.9069


## 15. Testing

In [ ]:
test_results = model.evaluate(
    [X_test_seq_s, X_test_know_s],
    {"state": Y_test_state, "duty": Y_test_duty, "reconstruction": X_test_know_s},
    verbose=1
)
print("\nTest results:")
for name, value in zip(model.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

state_prob, duty_pred, recon_pred = model.predict([X_test_seq_s, X_test_know_s])
state_pred = (state_prob > 0.5).astype(int)

1209/1209 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - duty_loss: 0.0496 - loss: 0.4206 - reconstruction_loss: 0.0065 - state_acc: 0.9166 - state_loss: 0.3690 - state_precision: 0.8359 - state_recall: 0.8196

Test results:
  loss: 0.4206
  compile_metrics: 0.3690
  state_loss: 0.0496
  duty_loss: 0.0065
  reconstruction_loss: 0.9166
1209/1209 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step


## 16. Saving Checkpoints

In [ ]:
final_model_path = os.path.join(CONFIG["checkpoint_dir"], "final_model.keras")
model.save(final_model_path)
print("Final model saved to:", final_model_path)
print("Best model (lowest val_loss) is saved at:", checkpoint_path)

Final model saved to: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2/final_model.keras
Best model (lowest val_loss) is saved at: /content/drive/MyDrive/SEM7PROJ/checkpoints_phase2/best_model.keras


## 17. Resume Training

In [ ]:
def resume_or_build_model():
    if os.path.exists(checkpoint_path):
        print("Existing checkpoint found. Loading model instead of training from scratch...")
        def weighted_bce_loader(pos_weights):
            pw = tf.constant(pos_weights, dtype=tf.float32)
            def loss_fn(y_true, y_pred):
                y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
                y_true = tf.cast(y_true, tf.float32)
                bce = -(pw * y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
                return tf.reduce_mean(bce)
            return loss_fn
        loaded_model = tf.keras.models.load_model(
            checkpoint_path,
            custom_objects={"loss_fn": weighted_bce_loader(pos_weights_arr)}
        )
        return loaded_model, True
    else:
        print("No checkpoint found.")
        return build_model(CONFIG["window_size"], KNOWLEDGE_DIM, len(CONFIG["appliances"]),
                            pos_weights_arr, CONFIG["recon_loss_weight"], CONFIG["duty_loss_weight"]), False

resumed_model, was_resumed = resume_or_build_model()
if was_resumed:
    model = resumed_model
    print("Resumed from checkpoint. Re-run Section 13's fit call manually if you want to continue training.")

Existing checkpoint found. Loading model instead of training from scratch...


ValueError: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().

## 18. Post-Hoc Minimum-Dwell-Time Correction ("Graph Reasoner")
A lightweight stand-in for a full appliance state-transition graph: removes
implausible single-window ON/OFF flicker by flipping isolated runs shorter than
`min_dwell_windows` to match their surrounding majority state. Windows must be in
time order for this to be meaningful.

In [ ]:
def apply_min_dwell_correction(state_sequence, min_dwell):
    """state_sequence: 1D binary array (time-ordered) for ONE appliance."""
    corrected = state_sequence.copy()
    n = len(corrected)
    i = 0
    while i < n:
        j = i
        while j < n and corrected[j] == corrected[i]:
            j += 1
        run_length = j - i
        if run_length < min_dwell and 0 < i and j < n:
            # isolated short run surrounded by the opposite state on both sides -> flip it
            if corrected[i - 1] == corrected[j] and corrected[i] != corrected[j]:
                corrected[i:j] = corrected[j]
        i = j
    return corrected

state_pred_corrected = state_pred.copy()
for i, app in enumerate(CONFIG["appliances"]):
    state_pred_corrected[:, i] = apply_min_dwell_correction(state_pred[:, i], CONFIG["min_dwell_windows"])

flips = int((state_pred_corrected != state_pred).sum())
print(f"Dwell-time correction flipped {flips} individual (window, appliance) predictions.")

Dwell-time correction flipped 704 individual (window, appliance) predictions.


## 19. Evaluation
Per-appliance precision/recall/F1 on the state head, before and after the dwell-time
correction, plus duration estimation (predicted vs ground-truth active minutes per
appliance) using the duty-cycle head — this avoids the duration-overcounting bug
described in the design report (crediting a whole window whenever any part was ON).

In [ ]:
print("=== State classification (raw model output) ===")
print(classification_report(Y_test_state, state_pred, target_names=CONFIG["appliances"], zero_division=0))

print("\n=== State classification (after dwell-time correction) ===")
print(classification_report(Y_test_state, state_pred_corrected, target_names=CONFIG["appliances"], zero_division=0))

for i, app in enumerate(CONFIG["appliances"]):
    p = precision_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    r = recall_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    f1 = f1_score(Y_test_state[:, i], state_pred_corrected[:, i], zero_division=0)
    print(f"{app:20s} | Precision: {p:.3f} | Recall: {r:.3f} | F1: {f1:.3f}")

# --- Duration estimation: predicted active minutes vs ground truth, using the duty head ---
window_minutes = CONFIG["window_size"] * 6 / 60.0  # window duration in minutes
print("\n=== Total active-time estimation (test split) ===")
print(f"{'Appliance':20s} {'Predicted (min)':>16s} {'Ground truth (min)':>20s} {'Error (min)':>14s}")
for i, app in enumerate(CONFIG["appliances"]):
    # predicted minutes = sum of (predicted duty fraction * window length), only counted
    # where the state head also predicts ON -- avoids crediting a full window just
    # because duty_pred is nonzero noise on an otherwise-OFF window
    pred_minutes = float(np.sum(duty_pred[:, i] * state_pred_corrected[:, i]) * window_minutes)
    true_minutes = float(np.sum(Y_test_duty[:, i]) * window_minutes)
    error = pred_minutes - true_minutes
    print(f"{app:20s} {pred_minutes:16.1f} {true_minutes:20.1f} {error:14.1f}")

=== State classification (raw model output) ===
                 precision    recall  f1-score   support

     dishwasher       0.50      0.98      0.66       948
washing machine       0.79      0.99      0.88      2798
         laptop       0.89      0.78      0.83     20429
         fridge       0.95      0.97      0.96     17180
  office lamp 1       0.44      0.41      0.42      5901

      micro avg       0.84      0.82      0.83     47256
      macro avg       0.71      0.83      0.75     47256
   weighted avg       0.84      0.82      0.83     47256
    samples avg       0.63      0.62      0.61     47256


=== State classification (after dwell-time correction) ===
                 precision    recall  f1-score   support

     dishwasher       0.51      0.98      0.67       948
washing machine       0.79      0.99      0.88      2798
         laptop       0.89      0.78      0.83     20429
         fridge       0.96      0.97      0.96     17180
  office lamp 1       0.44      0

## 20. Explainability Spot Checks
Manually inspect a true-positive and a true-negative prediction for one appliance to
confirm the model is distinguishing real activity from same-scale noise, not just
fitting spurious correlations.

In [ ]:
def spot_check(appliance, seq_raw, know_raw, y_true, y_pred_prob, n_examples=1):
    idx = CONFIG["appliances"].index(appliance)
    true_labels = y_true[:, idx]
    pred_probs = y_pred_prob[:, idx]

    tp_candidates = np.where((true_labels == 1) & (pred_probs > 0.5))[0]
    tn_candidates = np.where((true_labels == 0) & (pred_probs < 0.5))[0]

    print(f"--- {appliance}: true positive example ---")
    for idx_ex in tp_candidates[:n_examples]:
        w = seq_raw[idx_ex]
        print(f"  mean power {w.mean():.1f}W, max power {w.max():.1f}W "
              f"-> predicted ON at {pred_probs[idx_ex]:.2f} confidence. Correct.")

    print(f"--- {appliance}: true negative example ---")
    for idx_ex in tn_candidates[:n_examples]:
        w = seq_raw[idx_ex]
        print(f"  mean power {w.mean():.1f}W, max power {w.max():.1f}W "
              f"-> predicted OFF at {pred_probs[idx_ex]:.2f} confidence. Correct.")

for app in CONFIG["appliances"]:
    spot_check(app, X_test_seq, X_test_know, Y_test_state, state_prob)
    print()

--- dishwasher: true positive example ---
  mean power 1824.1W, max power 2167.0W -> predicted ON at 0.71 confidence. Correct.
--- dishwasher: true negative example ---
  mean power 152.4W, max power 171.0W -> predicted OFF at 0.21 confidence. Correct.

--- washing machine: true positive example ---
  mean power 1143.5W, max power 2167.0W -> predicted ON at 1.00 confidence. Correct.
--- washing machine: true negative example ---
  mean power 162.7W, max power 186.0W -> predicted OFF at 0.35 confidence. Correct.

--- laptop: true positive example ---
  mean power 162.7W, max power 186.0W -> predicted ON at 0.96 confidence. Correct.
--- laptop: true negative example ---
  mean power 30.6W, max power 121.0W -> predicted OFF at 0.35 confidence. Correct.

--- fridge: true positive example ---
  mean power 162.7W, max power 186.0W -> predicted ON at 0.92 confidence. Correct.
--- fridge: true negative example ---
  mean power 95.9W, max power 163.0W -> predicted OFF at 0.28 confidence. Correc